# Joint Inference v0.10 — One Temporal Parameter Across Multiple Mock Experiments
Synthetic data only. Five experiments have independent nuisance parameters but share one injected epsilon. Goal: test cross-system predictivity and compare epsilon=0 against epsilon!=0.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng=np.random.default_rng(11); t=np.linspace(0,10,801); n=len(t)
def sig(x): return 1/(1+np.exp(-x))
def make_profile(on,off,k,amp=1):
    W=sig(k*(t-on))*sig(k*(off-t)); dW=np.gradient(W,t)
    F=amp*(.30*W+.70*np.tanh(1.4*dW)); return W,F
specs=[('slow',2.5,7.5,2.5,1.0),('fast',3.2,6.8,10.,1.0),('asymmetric',2.0,7.0,5.,.72),('strong-nuisance',3.,7.,7.,1.15),('control',3.,7.,7.,0.)]
eps_true=1.8e-4; sigma=6e-5; datasets=[]
for m,(name,on,off,k,amp) in enumerate(specs):
    W,F=make_profile(on,off,k,amp); D=np.exp(-.5*((t-(4.4+.25*m))/(2.0+.1*m))**2); C=np.ones(n)
    beta=(1.0+0.45*m)*1e-4; drift=(-1)**m*(5+m)*1e-5; offset=(m-2)*1.2e-5
    y=eps_true*F+beta*W+drift*D+offset*C+rng.normal(0,sigma,n)
    datasets.append(dict(name=name,W=W,F=F,D=D,C=C,y=y))
print('experiments:',[d['name'] for d in datasets],' injected epsilon=',eps_true)


## Build joint design matrix
Column 0 is the one global epsilon template. Each experiment then receives its own interaction, drift and offset nuisance columns.

In [ ]:
M=len(datasets); rows=M*n; p=1+3*M; X=np.zeros((rows,p)); yall=np.zeros(rows)
for m,d in enumerate(datasets):
    sl=slice(m*n,(m+1)*n); X[sl,0]=d['F']; X[sl,1+3*m]=d['W']; X[sl,2+3*m]=d['D']; X[sl,3+3*m]=d['C']; yall[sl]=d['y']
cov=np.linalg.inv(X.T@X)*sigma**2; bhat=np.linalg.solve(X.T@X,X.T@yall); se=np.sqrt(np.diag(cov))
print('joint epsilon=',bhat[0],'+/-',se[0],' z=',bhat[0]/se[0],' true=',eps_true)
print('condition number=',np.linalg.cond(X))


## Null-vs-temporal model comparison
For known Gaussian sigma, compare chi-square and BIC. M1 adds one global epsilon parameter.

In [ ]:
def rss_fit(A,y):
    b=np.linalg.lstsq(A,y,rcond=None)[0]; r=y-A@b; return np.dot(r,r),b
rss1,b1=rss_fit(X,yall); X0=X[:,1:]; rss0,b0=rss_fit(X0,yall)
N=len(yall); k1=X.shape[1]; k0=X0.shape[1]
chi1=rss1/sigma**2; chi0=rss0/sigma**2; dchi=chi0-chi1
bic1=N*np.log(rss1/N)+k1*np.log(N); bic0=N*np.log(rss0/N)+k0*np.log(N)
print('Delta chi2 (M0-M1)=',dchi); print('Delta BIC (M0-M1)=',bic0-bic1)


## Leave-one-experiment-out prediction
Fit the shared epsilon on all but one experiment, then hold epsilon fixed while fitting only nuisance parameters in the held-out experiment.

In [ ]:
for hold in range(M):
    train=[i for i in range(M) if i!=hold]; nt=len(train)*n; Xt=np.zeros((nt,1+3*len(train))); yt=np.zeros(nt)
    for q,m in enumerate(train):
        d=datasets[m]; sl=slice(q*n,(q+1)*n); Xt[sl,0]=d['F']; Xt[sl,1+3*q]=d['W']; Xt[sl,2+3*q]=d['D']; Xt[sl,3+3*q]=d['C']; yt[sl]=d['y']
    et=np.linalg.lstsq(Xt,yt,rcond=None)[0][0]; d=datasets[hold]; Nh=np.column_stack([d['W'],d['D'],d['C']]); nuis=np.linalg.lstsq(Nh,d['y']-et*d['F'],rcond=None)[0]; pred=et*d['F']+Nh@nuis; rmse=np.sqrt(np.mean((d['y']-pred)**2))
    print('holdout',d['name'],' epsilon_train=',et,' RMSE=',rmse)


## Per-experiment epsilon consistency
A real common-law hypothesis should not require mutually incompatible epsilon values.

In [ ]:
ests=[]; errs=[]
for d in datasets:
    A=np.column_stack([d['F'],d['W'],d['D'],d['C']])
    if np.linalg.norm(d['F'])<1e-12:
        print(d['name'],'has no temporal sensitivity'); continue
    cv=np.linalg.inv(A.T@A)*sigma**2; b=np.linalg.solve(A.T@A,A.T@d['y']); ests.append(b[0]); errs.append(np.sqrt(cv[0,0])); print(d['name'],b[0],'+/-',np.sqrt(cv[0,0]))
plt.figure(figsize=(8,4.5)); plt.errorbar(range(len(ests)),ests,yerr=errs,fmt='o'); plt.axhline(eps_true,ls='--'); plt.xticks(range(len(ests)),[x['name'] for x in datasets if np.linalg.norm(x['F'])>=1e-12],rotation=30); plt.ylabel('epsilon'); plt.tight_layout(); plt.show()


## Injection-recovery null check
Regenerate data with epsilon=0 and verify that the same joint pipeline does not systematically create a nonzero shared temporal parameter.

In [ ]:
znull=[]
for rep in range(200):
    yn=np.zeros(rows)
    for m,d in enumerate(datasets):
        sl=slice(m*n,(m+1)*n); beta=(1.0+0.45*m)*1e-4; drift=(-1)**m*(5+m)*1e-5; offset=(m-2)*1.2e-5
        yn[sl]=beta*d['W']+drift*d['D']+offset*d['C']+rng.normal(0,sigma,n)
    bn=np.linalg.solve(X.T@X,X.T@yn); znull.append(bn[0]/se[0])
print('null z mean/std=',np.mean(znull),np.std(znull),' max |z|=',np.max(np.abs(znull)))


## Interpretation
Joint inference is stronger than fitting each curve independently because epsilon must explain several distinct identity-event shapes while nuisance amplitudes remain experiment-specific. The control has F=0 and therefore constrains nuisance behavior but cannot estimate epsilon itself. This remains a synthetic methodology study, not evidence for temporal physics.